# **Maestría en Inteligencia Artificial Aplicada**
## **Procesamiento de Lenguaje Natural PLN-NLP**

## Ejercicio para RAG con Base de Datos Vectorial con Chroma

#### Prof Luis Eduardo Falcón Morales

#### **Tecnológico de Monterrey**

* ### **Ejemplo de RAG con Chroma y fine-tuning**

* ### **Es un ejemplo muy sencillo, pero que ilustra la manera de guardar la información requerida para cuando un entrenamiento se interrumpe y poder continuar desde nos quedamos y para recuperar el modelo ajsutado para su uso en una nueva sesión.**

* ### **Recuerda que necesitas una GPU.**

* ### **RAG: Retrieval-Augmented Generation: Generación Aumentada por Recuperación**

# **PRIMERA PARTE: Configuración, Entrenamiento y Respaldo del proceso de Entrenamiento/Ajuste.**

## **1. Instlación y preparación de directorios**

In [1]:
!pip install -q chromadb sentence-transformers transformers accelerate bitsandbytes peft datasets

# Si indica warnings, puedes correrlo 2 veces para verificar que todo se instaló bien y reiniciar la sesión.

In [2]:
# Para cargar y la cuantización del modelo:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# En caso de incluir entrenamiento con ajuste fino de tu LLM:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

# Para los vectores embebidos y ChromaDB:
from sentence_transformers import SentenceTransformer

import chromadb


In [4]:
# Permitamos el acceso a tu Drive de Google para guardar ahí
# la información del modelo ajsutado

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
# Definimos las rutas de los directorios que vamos
# a requerir en nuestro Google Drive:

BASE_DIR = "/content/drive/MyDrive/miDIR_RAG01"

CHROMA_DIR = f"{BASE_DIR}/mi_chromadb"
CHECKPOINTS_DIR = f"{BASE_DIR}/checkpoints"
MODELO_FINAL_DIR = f"{BASE_DIR}/modelo_final_rag"


In [6]:
import os

os.makedirs(CHROMA_DIR, exist_ok=True)
os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
os.makedirs(MODELO_FINAL_DIR, exist_ok=True)


In [7]:
# Podemos usar comandos de Linux para movernos entre
# las diferentes carpetas/directorios de nuestro Drive:
# https://documentation.ubuntu.com/desktop/en/latest/tutorial/the-linux-command-line-for-beginners/?_gl=1*etr8va*_gcl_au*MTMxMDIxMzY3MC4xNzgwNTc4Njkz

# listar/directorio
!ls

drive  sample_data


In [8]:
# print working directory
!pwd

/content


In [ ]:
# change directory.
# % automagic reconoce el comando de Linux
# Con "cd .." puedes salir de dicho directorio.

#%cd /content/drive/MyDrive/

/content/drive/MyDrive


In [ ]:
# make directory
#!mkdir miDir_RAG01

In [ ]:
# Para entrar a un directorio/carpeta particular:
#%cd miDir_RAG01

/content/drive/MyDrive/miDir_RAG01


## **2. Documentos, Embeddings y ChromaDB**

In [9]:
# Por el momento supongamos que este/estos son nuestros PDFs a
# partir del cual vamos a contruir nuestra base de datos vectorial:

documentos = [
    "La Inteligencia Artificial está transformando las industrias de manera global.",
    "El cambio climático es uno de los grades retos de nuestro tiempo.",
    "Los Modelos de Lenguaje de Gran Tamaño pueden generar texto de manera muy parecida a los humanos.",
    "Entre las fuentes de energía renovable se encuentran la energía solar y la eólica.",
    "El Procesamiento de Lenguaje Natural (PLN o NLP) es una área de la Inteligencia Artificial."
]

In [10]:
# Generamos los chunks de nuestros documentos.
# Podrás definir una mejor función para documentos más complejos:

def chunk_text(text, chunk_size=50, overlap=10):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size-overlap)]

chunks = []
for doc in documentos:
    chunks.extend(chunk_text(doc))


In [11]:
len(chunks)

13

In [12]:
chunks[0]

'La Inteligencia Artificial está transformando las '

In [13]:
chunks[1]

'mando las industrias de manera global.'

### **Algunas recomendaciones para vectores embebidos de HuggingFace para español:**

**Existen muchas opciones y cada cierto tiempo aparecen nuevas, pero por el momento puedes probar algunas de las siguientes de HuggingFace:**

  * **intfloat/multilingual-e5-small** Puedes también probar el base y el large, todos bastante pequeños (0.1B, 0.3B, 0.6B) y de dimensiones 384, 768, 1024, respectivamente. Para muchos idiomas.

  * **jinaai/jina-embeddings-v2-base-es** Para español e inglés. De 0.2B de parámetros y dimensiones de 768. Ya puedes usarlo con textos más largos de hasta 8K tokens.

  * **BAAI/bge-m3** Muchos idiomas. Modelo pequeño de 570M de parámetros pero con vectores de dimensión 1024.

  * **sentence-transformers/paraphrase-multilingual-mpnet-base-v2** Muchos idiomas, de 0.3B de parámetros y vectores de dimensión 768.


In [14]:
# Obtengamos los vectores embebidos:

embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [15]:
# Se usa PersistentClient para guardar la informacion en el Drive:

client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_or_create_collection(name="rag_demo")

In [16]:
# Hay que verificar que la colección esté vacía para evitar duplicados.
# Cada vez que usamos collection.add() de manera automática ChromaDB actualiza
# la colección en nuestro directorio del Google-Drive:

if collection.count() == 0:

    # Obtenemos los embeddins de cada chunk:
    embeddings = embedding_model.encode(chunks).tolist()

    # Los guardamos en nuestro directorio:
    collection.add(
        documents=chunks,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(chunks))]
    )
    print(f"Se añadieron {collection.count()} documentos a ChromaDB en Drive.")
else:
    print(f"ChromaDB ya tenía {collection.count()} documentos. Se omite la inserción.")


ChromaDB ya tenía 12 documentos. Se omite la inserción.


In [17]:
# Definimos la función que nos ayudará a recuperar (retrieve) mediante
# búsqueda semántica los top-k vectores embebidos similares dentro de
# la base de datos vectorial previamente construida con nuestros documentos:

def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    return results["documents"][0]

## **3. Modelos, cuantizados-Q y con LoRA (en caso de incluir ajuste fino).**

**Ya hemos usado varios LLM para español, pero aquí recordemos algunas otras. Igualmente sabemos que estarán apareciendo constantemente nuevas y mejores opciones, por lo que no dejes de seguir revisando la plataforma de HF:**

* **Un modelo LLM "base" generador de texto es aquel que digamos solo es capaz de completar texto.**

* **Modelos LLM "Instruct", son aquellos modelos base que además han sido ajustados para seguir instrucciones o mantener un diálogo.**

* **Los modelos "LLM-Instruct" son en general la recomendación inicial para aplciarse en los sistemas RAG, complementando obviamente con algún modelo de vectores embebidos adecuado.**

* **Algunas recomendaciones de buenos LLMs que en general podamos usar con idioma español son los siguientes (aunque también podrían requerir cuantización para trabajar con ellos en una computadora personal o para abaratar costos de uso):**

  * **meta-llama/Llama-3.1-8B-Instruct** LLM de uso general (chat, RAG, razonamiento, etc) multilingüe de 8B.

  * **Qwen/Qwen2.5-7B-Instruct** LLM de uso general (chat, RAG, razonamiento, etc) multilingüe de 7B.

  * **google/gemma-2-9b-it** LLM de uso general (chat, RAG, razonamiento, etc) multilingüe de 9B. Recomendable además para generación de contenido creativo y de marketing. Igualmente está ajustado para instrucciones: it (instruction tuned).

  * **mistralai/Mistral-7B-Instruct-v0.3** LLM de uso general (chat, RAG, razonamiento, etc) multilingüe de 7B.

  ---

* **Algunos más pequeños que puedas usar rápidamente sin cuantización para usar en tu PC o para soluciones en móviles o de la llamada IA de borde (AI-edge: sensores IoT, cámaras, celulares, vehículos, etc.):**

  * **CEGTEdicion/Llama-3.2-1B-espanol** Ajustado para que el lenguaje en español sea más fluido. De 1B. A diferencia de los modelos Gemma, Qwen o los mismos Llama multilingüe,este modelo está ajustado para hispanoparlantes.

  * **Kukedlc/NeuralGemma2-2b-Spanish** Modelo de Google resultado de la fusión de otros dos modelos para mejorar la fluidez del español y entregar un buen equilibrio entre calidad y eficiencia computacional. De 2B.

  * **meta-llama/Llama-3.2-1B-Instruct** Versión ligera de la familia Llama-3.x. Multilingüe.

  * **Qwen/Qwen2.5-1.5B-Instruct** La versión ligera de 1.5B del modelo de 7B. Multilingüe.

  * **HuggingFaceTB/SmolLM2-1.7B-Instruct** Modelo muy ligero y especialmente diseñado con fines educativos con ajuste fino para conocimiento general, razonamiento, matemáticas y generación de código. Tiene además versiones más ligeras de 135M y 360M para utilizar en diferentes ciclos escolares donde los estudiantes tengan recursos más limitados y el conocimiento o reazonamiento no sea tan complejo. Multilingüe.

  * **microsoft/Phi-3-mini-4k-instruct** Modelo intermedio de 3.8B con una calidad superior en razonamiento a los anteriores, con su adicional costo de recursos computacionales. Multilingüe.

---

* **Recuerda que de manera general modelos de tamaño xB en formato FB16 requieren aproximadamente 2*xGB de VRAM en GPU. Y con cuantización 4-bit aproximadamente un poco menos de xGB.**

  * **Así, 1B requieren VRAM de 2GB en FB16 y de 1GB con cuantizado 4-bit. Modelos de de 7-8B en FB16 requieren aproximadamente 16GB de VRAM y con cuantización 4-bit, solo unos 6GB. Modelos de 9B requieren unos 20GB de VRAM y con cuantización 4-bit unos 8GB.**

  * **A lo anterior en general habría que añadir otros 2-4GB adicionales de VRAM debido a la longitud de contexto utilizada, el tamaño de batch, la cuantización utilizada, el manejo de información entre CPU/GPU, etc.**

  ---






In [18]:
# Usemos por el momento un LLM ligero:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# Nuestros tokens:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Asegurar que el tokenizer tenga pad_token, que algunos modelos no lo incluyen
# y se requiere durante el entrenamiento.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [19]:
# Configuramos la cuantización-Q a 4-bit y lo aplicamos
# al modelo LLM seleccionado:

# Configuramos los argumentos a formato 4-bit:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### **En caso de incluir ajuste fino de tu LLM**

In [20]:
# Si el problema incluye ajuste fino del modelo, tendríamos además
# este apartado con LoRA para las matrices indicadas:

# Preparar modelo para entrenamiento k-bit y añadir adaptadores LoRA
model = prepare_model_for_kbit_training(model)
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],   # ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# Recuerda que en general deberás tener menos del 1% para parámetros entrenables:
model.print_trainable_parameters()



trainable params: 1,089,536 || all params: 1,544,803,840 || trainable%: 0.0705


### **Conjunto de entrenamiento:**

In [21]:
# Preparamos un conjunto de entrenamiento.
# En este ejemplo pequeño contruimos uno sintético
# de los mismos documentos que tenemos, solo para
# los propósitos de este ejercicio. Pero aquí debieras
# usar obviamente los de train correspondientes.

from datasets import Dataset

# Creamos un dataset de ejemplo simple basado en tus documentos
train_samples = []
for doc in documentos:
    train_samples.append({
        "text": f"Contexto: {doc}\nPregunta: ¿De qué trata este texto?\nRespuesta: Este texto trata sobre {doc[:30]}..."
    })

dataset = Dataset.from_list(train_samples)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=256, padding="max_length")

tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=["text"])


Map:   0%|          | 0/5 [00:00<?, ? examples/s]

### **Configuración del modelo y entrenamiento.**

In [25]:
# Configuramos el entrenamiento del modelo, indicando la manera
# en que se guardarán los Checkpoints en tu Drive, en caso de perder
# la sesión de entrenamiento, o al final, para recuperar los pesos ajustados.

training_args = TrainingArguments(
    output_dir=CHECKPOINTS_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3, # 1 o 2 épocas. Si es más grande debes validar que no pierde poder el LLM.

    #evaluation_strategy="steps",  # Dependiendo la versión de "transformers" que tengas instalada
    eval_strategy="no",   # Usar "steps" si tienes conjunto de validación para validar.
    eval_steps=2,         # Generalmente mayor que logging_steps por el costo computacional. Se puede usar 500 para procesos largos.

    save_strategy="steps",  # Para definir la estrategia de cada cuando estará guardando los Checkpoints.
    save_steps=2,          # Guarda un checkpoint cada 2 pasos. En general puede ser 500: checkpoints/checkpoint-500/. Usar igual que eval_steps en general.
    save_total_limit=2,     # Mantiene solo los últimos 2 checkpoints para no saturar el Drive.
    logging_steps=1,        # Reporta errores de entrenamiento, tamaño de paso, métricas, etc, para mostrar durante el progreso. No incluye Val. Usualmente menor que eval_steps porque es menos costoso.
    report_to="none", # Si deseas monitorear el entrenamiento en una plataforma externa: TensorBoard, Weights & Biases (wandb), ...
    fp16=True,
)

# Listo, conjuntamos el modelo con los argumentos definidos para su entrenamiento:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)


# Antes de pasar al entrenmiento, verificamos si estamos iniciando desde cero,
# o desde alguna sesión interrumpida. Buscamos en tu directorio de Checkpoints del Drive:
checkpoints = [d for d in os.listdir(CHECKPOINTS_DIR) if d.startswith("checkpoint-")]
resume_from_checkpoint = None

if checkpoints:
    # Ordenamos para obtener el checkpoint más reciente (el número más alto)
    checkpoints.sort(key=lambda x: int(x.split("-")[-1]))
    resume_from_checkpoint = os.path.join(CHECKPOINTS_DIR, checkpoints[-1])
    print(f"Checkpoint encontrado. Reanudando entrenamiento desde: {resume_from_checkpoint}")
else:
    print("No se encontraron checkpoints. Iniciando entrenamiento desde cero.")


# Ahora sí, realizamos el entrenamiento desde el checkpoint identificado:
trainer.train(resume_from_checkpoint=resume_from_checkpoint)


Checkpoint encontrado. Reanudando entrenamiento desde: /content/drive/MyDrive/miDIR_RAG01/checkpoints/checkpoint-6


Step,Training Loss


TrainOutput(global_step=6, training_loss=0.0, metrics={'train_runtime': 0.0028, 'train_samples_per_second': 5390.212, 'train_steps_per_second': 2156.085, 'total_flos': 30215350517760.0, 'train_loss': 0.0, 'epoch': 3.0})

## **4. Validación del modelo antes de guardar la versión final**

### **Verifiquemos el modelo ajustado cómo se comporta:**

In [26]:
# Definimos nuestro prompt que mandará información al LLM de lo recuperado
# de la base de datos vectorial mediante búsqueda semántica:

def build_prompt(contexto, pregunta):
    txt_contexto = "\n\n".join(contexto)

    prompt = f"""
Eres un asistente que responde a preguntas usando solamente lo que se te proporciona en el contexto.
Si no encuentras información de algún tema en el contexto, comenta que no sabes.

Contexto:
{txt_contexto}

Pregunta:
{pregunta}

Respuesta:
"""
    return prompt

In [27]:
# Verifiquemos que el modelo se comporta como debiera ser.
# Definimos la función que hace la búsqueda semántica en la
# base de datos vectorial y responde con ello el LLM:

def rag_answer(pregunta):
    contexto = retrieve(pregunta, k=3)   # Recuperamos las k similaridades de la bd vectorial

    prompt = build_prompt(contexto, pregunta)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.3,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer.split("Respuesta:")[-1].strip()


In [28]:
# En este ejemplo en particular es obvio que los pocos documentos no son
# suficientes para un buen resultado y la salida seguramente no será
# satisfactoria, pero con esto completamos el ciclo para verificar que
# tu modelo está listo para usarse y procedemos a guardar la información
# correspondiente para usarlo posteriormente en producción.

input = "¿Qué es el procesamiento del lenguaje natural?"
print(rag_answer(input))

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


El procesamiento del lenguaje natural es una área dentro de la inteligencia artificial. Los modelos de gran tamaño pueden generar grandes cantidades de texto y son utilizados para mejorar la IA. La IA está cambiando cómo interactúan los seres humanos con la tecnología. Esta área también implica la creación de sistemas de aprendizaje automático. Estas tecnologías están siendo desarrolladas para automatizar tareas repetitivas y para ayudar al usuario a realizar t


### **Por último, guardamos la información del modelo final.**

In [29]:
# Suponiendo que ya tienes el modelo final ajsutado, procedemos
# a guardarlo, junto con los tokens, para usarlo en sesiones futuras.

print("Guardando el modelo final en el Google-Drive...")
trainer.save_model(MODELO_FINAL_DIR)
tokenizer.save_pretrained(MODELO_FINAL_DIR)

print(f"\nModelo y tokenizer guardados exitosamente en:\n{MODELO_FINAL_DIR}")


Guardando el modelo final en el Google-Drive...

Modelo y tokenizer guardados exitosamente en:
/content/drive/MyDrive/miDIR_RAG01/modelo_final_rag


---

## **Fin de la Primera Parte**

---

# **SEGUNDA PARTE: Nueva sesión para uso del modelo ya entrenado/ajustado.**

* ### **Esta segunda parte simula que retomas la sesión después de haber terminado y cerrado la primera parte, por lo que puedes reiniciarla para verificar que en efecto todo funciona a continuación.**

* ### **Recuerda que en este ejercicio utilizamos documentos sintéticos muy simples, por lo que ni los datos son suficientes para que aprenda algo, ni tampoco para que pueda hacer el ajuste fino adecuado. Es decir, no esperes que los resultados que veas a continuación sean muy precisos.**

* ### **Recuerda usar GPU. Aunque en este caso puedes usar tal vez una más sencilla que la usada durante el entrenamiento/ajuste y en caso de tener opciones de selección.**

In [30]:
# Instalamos lo necesario:
!pip install -q chromadb sentence-transformers transformers accelerate bitsandbytes peft

In [ ]:
# Tu acceso al Drive:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [31]:
# Las rutas de tus directorios.
# Observa que ya no necesitamos el de los checkpoints.

BASE_DIR = "/content/drive/MyDrive/miDIR_RAG01"
CHROMA_DIR = f"{BASE_DIR}/mi_chromadb"
MODELO_FINAL_DIR = f"{BASE_DIR}/modelo_final_rag"


### **ChromaDB y Embeddings:**

In [32]:
# Cargamos la información guardada por ChromaDB en nuestro directorio.
# Es decir, los vectores embebidos de la base de datos vectorial que
# se construyó:

import chromadb
from sentence_transformers import SentenceTransformer

# Observa que debe ser el mismo modelo con el que generaste
# los embebidos en la primera parte:
modelo = "intfloat/multilingual-e5-small"

# Cargamos el Cliente Persistente para que lea lo que guardamos:
client = chromadb.PersistentClient(path=CHROMA_DIR)
collection = client.get_collection("rag_demo")

embedding_model = SentenceTransformer(modelo)

# Una vez con nuestro modelo de embeddings, definimos la función que
# nos ayudará a recuperar los k más cercanos:
def retrieve(query, k=3):
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(query_embeddings=query_embedding, n_results=k)
    return results["documents"][0]


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

### **Modelo LLM y su ajuste de pesos guardados en el Drive de haber hecho ajuste-fino con LoRA**

In [33]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Si hiciste ajuste fino al modelo en la primera parte, tendrás que
# cargar aquí el mismo modelo. Si no hiciste ajuste fino, entonces
# podrías en dado caso usar algún otro LLM:
model_name_base = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Cargamos el tokenizer desde nuestro Drive:
tokenizer = AutoTokenizer.from_pretrained(MODELO_FINAL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 2. Configuramos la cuantización 4-bit:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Cargamos ahora el modelo base cuantizado:
model = AutoModelForCausalLM.from_pretrained(
    model_name_base,
    device_map="auto",
    quantization_config=bnb_config
)

# 4. Finalmente, inyectamos  al modelo LLM base los pesos ajustados
#    con LoRA en las matrices A y B de LoRA que configuramos y que
#    teníamos guardos en Google-Drive:
model = PeftModel.from_pretrained(model, MODELO_FINAL_DIR)
print("Modelo ajustado (fine-tuned) cargado exitosamente desde Google Drive.")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Modelo ajustado (fine-tuned) cargado exitosamente desde Google Drive.


### **De nuevo nuestra función de recuperación RAG y prueba:**

In [34]:
# De nuevo nuestra función propmpt:

def build_prompt(contexto, pregunta):
    txt_contexto = "\n\n".join(contexto)
    return f"""
Eres un asistente que responde a preguntas usando solamente lo que se te proporciona en el contexto.
Si no encuentras información de algún tema en el contexto, comenta que no sabes.

Contexto:
{txt_contexto}

Pregunta:
{pregunta}

Respuesta:
"""



# Y la función de recuperación de información:

def rag_answer(pregunta):
    contexto = retrieve(pregunta, k=3)
    prompt = build_prompt(contexto, pregunta)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad(): # Aquí hay que desactivar los gradientes para inferencia, pues ya no se está ajsutando.
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            temperature=0.3,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )

    # Retomando el texto desde los embeddings:
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Y recuperamos la respuesta:
    return answer.split("Respuesta:")[-1].strip()


In [35]:
# Probemos con una nueva pregunta:
pregunta_usuario = "¿Cuáles son las fuentes de energía renovable mencionadas?"
respuesta = rag_answer(pregunta_usuario)

print(f"Pregunta: {pregunta_usuario}")
print(f"Respuesta del Modelo: {respuesta}")

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Pregunta: ¿Cuáles son las fuentes de energía renovable mencionadas?
Respuesta del Modelo: La respuesta es: La energía solar y la eólica.


---

## **Fin de la Segunda Parte**

---